# references.json 방출기 테스트 (작업지시서 §4.5)

앞 단계(`cells.jsonl`)가 **"각 칸에 뭐가 들어있나"**의 목록이었다면,
이번 `references.json`은 **"칸들이 서로 어떻게 연결돼 있나"**의 지도입니다.

예를 들어 어떤 칸의 수식이 `='1100'!B4` 라면 —
"이 칸은 1100 시트의 B4 칸을 가져다 쓴다"는 **화살표(edge)** 하나가 생깁니다.
이 노트북은 그 화살표들을 뽑아내는 과정이 제대로 되는지 눈으로 확인합니다.

확인하는 것:
1. 수식 한 줄에서 참조를 어떻게 뽑는지 (파서 맛보기)
2. **골든 6엣지** — 감사계약 파일에서 지시서가 정한 정답 화살표 6개가 나오는지
3. impacts — 화살표를 거꾸로 뒤집은 "누가 나를 참조하나" 색인
4. 까다로운 경우들 — 외부 파일 참조·INDIRECT·함수 이름 함정
5. xls 파일 — 수식을 못 읽는 경우의 정직한 표시
6. 결정론(P2) — 두 번 만들어도 똑같은가 + 32종 전수

> 실행 전 커널이 이 프로젝트의 `.venv`인지 확인하세요.

In [1]:
# 0. 준비 — 경로 자동 판별과 import
from pathlib import Path
import hashlib, json, sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():   # tests/ 안에서 열었을 때
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from excel_to_skill.extractor import extract_workbook, WorkbookIR, SheetIR, CellIR
from excel_to_skill.refparse import parse_formula
from excel_to_skill.emit_refs import build_references, write_references, mask_pii

RAW = ROOT / "workingpaper_raw"   # 실조서 코퍼스 (git 밖)
OUT = ROOT / "tests" / "_output"  # 노트북 산출물 (git 밖)
OUT.mkdir(exist_ok=True)
print("프로젝트 루트:", ROOT)
print("코퍼스 파일 수:", len(list(RAW.glob("*.xls*"))))

프로젝트 루트: /home/shin/Project/Excel_to_Skill
코퍼스 파일 수: 32


## 1. 수식 한 줄에서 참조 뽑기 (파서 맛보기)

`parse_formula`는 수식 글자를 훑어서 "어느 시트의 어느 칸을 가리키나"를 찾아냅니다.
완전한 수식 계산기가 아니라 **주소 모양을 골라내는** 수준입니다.

각 참조에서 보는 것:
- `sheet` — 가리키는 시트 (`None`이면 수식이 있는 그 시트 자신)
- `coord` — 칸 주소. `$B$4`의 `$`는 떼고 `B4`로 **정리**하지만,
- `raw` — 수식에 적힌 **원문 그대로** ($·따옴표·`[n]` 다 보존)
- `ref_type` — 칸 하나면 `cell`, `A1:B10`처럼 범위면 `range` (범위는 안 쪼갬)

In [2]:
examples = [
    "='1100'!B4",              # 다른 시트의 한 칸
    "=SUM(A1:B10)*2",          # 범위 하나 (200칸으로 안 쪼갬)
    "='시트 이름'!$C$3",         # 공백 든 시트명 + 절대참조
    "=[2]2600!A1",             # 외부 파일 참조 ([2] = 링크테이블 2번)
    "=LOG10(B2)",              # 함정: LOG10은 함수, B2만 참조
    '=IF(A1="B2",C1,D1)',      # 함정: 따옴표 안 "B2"는 글자, 무시
]
for f in examples:
    body = f[1:] if f.startswith("=") else f   # 보기용으로 = 떼기
    toks = parse_formula(body)
    print(f"{f:28} ->")
    for t in toks:
        sheet = t.sheet if t.sheet else "(자기 시트)"
        print(f"      {sheet} / {t.coord} [{t.ref_type}]  원문={t.raw!r}")
    if not toks:
        print("      (참조 없음)")

='1100'!B4                   ->
      1100 / B4 [cell]  원문="'1100'!B4"
=SUM(A1:B10)*2               ->
      (자기 시트) / A1:B10 [range]  원문='A1:B10'
='시트 이름'!$C$3                ->
      시트 이름 / C3 [cell]  원문="'시트 이름'!$C$3"
=[2]2600!A1                  ->
      2600 / A1 [cell]  원문='[2]2600!A1'
=LOG10(B2)                   ->
      (자기 시트) / B2 [cell]  원문='B2'
=IF(A1="B2",C1,D1)           ->
      (자기 시트) / A1 [cell]  원문='A1'
      (자기 시트) / C1 [cell]  원문='C1'
      (자기 시트) / D1 [cell]  원문='D1'


## 2. 골든 6엣지 — 정답과 대조

지시서 §4.5는 감사계약 파일(`1100~1300`)에서 **정확히 이 6개 화살표**가
나와야 한다고 못박아 뒀습니다(검증 항목 V5). 하나라도 빠지거나 더 나오면 실패입니다.

- `1200!B4 → 1100!B4` : 1200 시트가 1100의 값을 그대로 물려받음
- `1200!D4 → 1100!D4`
- `1200!B5 → 1100!B5`
- `1300!B4 → 1100!B4`
- `1300!D4 → 1200!D4` : 1300은 1200을 거쳐 물려받기도 함
- `1300!B5 → 1100!B5`

In [3]:
gyeyak = next(RAW.glob("*1100~1300*"))
ir = extract_workbook(gyeyak)
doc = write_references(ir, OUT / "감사계약.references.json")

GOLDEN = [
    ("1200!B4", "1100!B4"), ("1200!D4", "1100!D4"), ("1200!B5", "1100!B5"),
    ("1300!B4", "1100!B4"), ("1300!D4", "1200!D4"), ("1300!B5", "1100!B5"),
]
got = [(e["from"], e["to"]) for e in doc["edges"]]

print(f"원본: {gyeyak.name}")
print(f"방출된 화살표 {len(got)}개:\n")
for e in doc["edges"]:
    print(f"  {e['from']:>8}  →  {e['to']:<8}   (수식: {e['formula']}, {e['ref_type']})")

missing = [g for g in GOLDEN if g not in got]
extra = [g for g in got if g not in GOLDEN]
print()
print("빠진 정답:", missing if missing else "없음")
print("불필요한 초과:", extra if extra else "없음")
print("\nV5 골든 판정:", "통과 ✓" if not missing and not extra else "실패 ✗")

원본: 감사조서서식_1100~1300 감사계약.xlsx
방출된 화살표 6개:

   1200!B4  →  1100!B4    (수식: '1100'!B4, cell)
   1200!D4  →  1100!D4    (수식: '1100'!D4, cell)
   1200!B5  →  1100!B5    (수식: '1100'!B5, cell)
   1300!B4  →  1100!B4    (수식: '1100'!B4, cell)
   1300!D4  →  1200!D4    (수식: '1200'!D4, cell)
   1300!B5  →  1100!B5    (수식: '1100'!B5, cell)

빠진 정답: 없음
불필요한 초과: 없음

V5 골든 판정: 통과 ✓


## 3. impacts — 화살표를 거꾸로 뒤집기

`edges`가 "A는 B를 참조한다"라면, `impacts`는 그걸 뒤집어
**"B를 참조하는 애들은 누구누구냐"**를 미리 모아둔 색인입니다.

나중에 스킬 에이전트가 "1100!B4 값을 고치면 어디에 영향이 가지?"라고 물었을 때
바로 답할 수 있게 하는 재료입니다. edges에서 기계적으로 뒤집어 만들 뿐 새 정보는 아닙니다.

In [4]:
print("impacts (누가 이 칸을 참조하나):\n")
for target, sources in doc["impacts"].items():
    print(f"  {target:>8}  ←  {', '.join(sources)}")

# 검산: impacts를 다시 뒤집으면 edges 개수와 맞아야 함
flat = sum(len(v) for v in doc["impacts"].values())
print(f"\nimpacts 항목 총합 {flat}개 = edges {len(doc['edges'])}개  ->",
      "일치 ✓" if flat == len(doc["edges"]) else "불일치 ✗")

impacts (누가 이 칸을 참조하나):

   1100!B4  ←  1200!B4, 1300!B4
   1100!B5  ←  1200!B5, 1300!B5
   1100!D4  ←  1200!D4
   1200!D4  ←  1300!D4

impacts 항목 총합 6개 = edges 6개  -> 일치 ✓


## 4. 까다로운 경우들 — 합성 파일로 확인

실제 감사조서 4,048개 수식에는 외부 파일 참조도 INDIRECT도 **하나도 없었습니다**.
그래서 이 경우들은 **가짜 통합문서(IR)를 손으로 만들어** 확인합니다.

- **외부 파일 참조** `[1]요약!B2` : 링크 테이블 1번과 연결(조인)해서 원본 위치를 붙임.
  단, 그 경로에 이메일이 있으면 `k***@` 로 **가림**(개인정보 규칙 P7).
- **없는 링크 번호** `[9]…` : 링크 테이블에 9번이 없으면 → 억지로 지어내지 않고 `null`(모름).
- **INDIRECT / OFFSET** : 값을 실제로 계산해봐야 목적지를 아는 수식.
  풀 수 없으니 **화살표로 만들지 않고** `unresolved`(미해결) 목록에 정직하게 남김.
  단 `OFFSET(A1, ...)`의 `A1`처럼 **눈에 보이는 주소**는 관찰된 사실이므로 화살표로도 남깁니다.

In [5]:
s = SheetIR(name="본문", index=0)
s.cells[(1, 1)] = CellIR(row=1, col=1, formula="[1]요약!B2+A9")      # 외부참조 + 자기시트 A9
s.cells[(2, 1)] = CellIR(row=2, col=1, formula="'[9]없는링크'!C3")     # 링크 번호 초과
s.cells[(3, 1)] = CellIR(row=3, col=1, formula='INDIRECT("본문!A"&B1)')  # 동적 -> 미해결
s.cells[(4, 1)] = CellIR(row=4, col=1, formula="OFFSET($A$1,1,2)")     # 동적이나 A1은 정적

fake = WorkbookIR(
    source_path=Path("가짜.xlsx"), format="xlsx", loader_path="openpyxl_normal",
    sheets=[s],
    external_links=["file:///서버/조서/kim.lee@firm.co.kr/원본.xlsx"],  # 이메일 포함 경로
)
fdoc = build_references(fake)

print("── 외부 파일 참조 (external_refs) ──")
for r in fdoc["external_refs"]:
    print(f"  {r['cell']}: 원문={r['raw']!r}")
    print(f"         연결된 원본 = {r['target']}")
print("  ↑ 이메일 kim.lee@ 가 k***@ 로 가려졌는지, 없는 9번은 null 인지 확인\n")

print("── 미해결 (unresolved) ──")
for u in fdoc["unresolved"]:
    print(f"  {u['cell']}: {u['formula']}  → 이유: {u['reason']}")

print("\n── 그래도 남는 정적 화살표 (edges) ──")
for e in fdoc["edges"]:
    print(f"  {e['from']} → {e['to']}   (수식: {e['formula']})")
print("  ↑ OFFSET 안의 A1, [1]수식 안의 A9 는 화살표로 남음")

── 외부 파일 참조 (external_refs) ──
  본문!A1: 원문='[1]요약!B2'
         연결된 원본 = file:///서버/조서/k***@firm.co.kr/원본.xlsx
  본문!A2: 원문="'[9]없는링크'!C3"
         연결된 원본 = None
  ↑ 이메일 kim.lee@ 가 k***@ 로 가려졌는지, 없는 9번은 null 인지 확인

── 미해결 (unresolved) ──
  본문!A3: INDIRECT("본문!A"&B1)  → 이유: indirect
  본문!A4: OFFSET($A$1,1,2)  → 이유: offset

── 그래도 남는 정적 화살표 (edges) ──
  본문!A1 → 본문!A9   (수식: [1]요약!B2+A9)
  본문!A3 → 본문!B1   (수식: INDIRECT("본문!A"&B1))
  본문!A4 → 본문!A1   (수식: OFFSET($A$1,1,2))
  ↑ OFFSET 안의 A1, [1]수식 안의 A9 는 화살표로 남음


## 5. xls 파일 — 못 읽으면 못 읽었다고

옛날 `.xls` 형식은 라이브러리(xlrd)가 **수식 원문을 못 읽습니다**.
이럴 때 화살표를 0개로 두고 **"안 나온 게 아니라 못 읽은 것"**임을
`observability`에 `unavailable_xls`로 표시합니다(관찰 3상태 P6).
빈 결과와 "관찰 불가"를 구분하는 게 핵심입니다.

In [6]:
xls_path = next(RAW.glob("*7540*"))   # .xls 파일
xls_doc = build_references(extract_workbook(xls_path))
print(f"파일: {xls_path.name}")
print(f"edges: {len(xls_doc['edges'])}개")
print(f"observability: {json.dumps(xls_doc['observability'], ensure_ascii=False)}")
print("\n↑ edges 0개지만 'full'이 아니라 'unavailable_xls' — 못 읽었다는 뜻")

파일: 감사조서서식_7540 연결공시사항점검표 (KIFRS용)_2025.xls
edges: 0개
observability: {"workbook": "unavailable_xls", "note": "xls(xlrd): 수식 원문 접근 불가 — 참조 그래프 관찰 불가(P6). 외부 링크 목록 관찰 불가."}

↑ edges 0개지만 'full'이 아니라 'unavailable_xls' — 못 읽었다는 뜻


## 6. 결정론(P2) + 32종 전수

같은 파일을 두 번 처리해서 결과가 **글자 하나까지 똑같은지**(sha256 해시 비교),
그리고 코퍼스 32종 전체가 실패 없이 방출되는지 봅니다.
이것이 나중에 `verify` 명령의 V3(재현성)·V5(골든) 검사가 할 일의 예행연습입니다.

In [7]:
print(f"{'파일':<44} {'edges':>6} {'ext':>4} {'미해결':>5} {'관찰':<16} 재현")
fails, tot = [], 0
for p in sorted(RAW.glob("*.xls*")):
    try:
        d1 = build_references(extract_workbook(p))
        d2 = build_references(extract_workbook(p))
        b1 = json.dumps(d1, ensure_ascii=False).encode()
        b2 = json.dumps(d2, ensure_ascii=False).encode()
        same = hashlib.sha256(b1).hexdigest() == hashlib.sha256(b2).hexdigest()
        tot += len(d1["edges"])
        if not same:
            fails.append((p.name, "결정론 불일치"))
        print(f"{p.name[:42]:<44} {len(d1['edges']):>6} {len(d1['external_refs']):>4} "
              f"{len(d1['unresolved']):>5} {d1['observability']['workbook']:<16} {'✓' if same else '✗'}")
    except Exception as e:
        fails.append((p.name, repr(e)))
        print(f"{p.name[:42]:<44} 실패: {e!r}")
print(f"\n화살표 합계 {tot}개, 실패 {len(fails)}건")
for name, err in fails:
    print(f"  ✗ {name}: {err}")

파일                                            edges  ext   미해결 관찰               재현
감사조서서식_01 조서파일 KIFRS.xlsx                         0    0     0 full             ✓
감사조서서식_02 영구조서목록.xlsx                             0    0     0 full             ✓
감사조서서식_03 일반조서목록 KIFRS.xlsx                       0    0     0 full             ✓
감사조서서식_04 감사조서철 작성 및 보존.xlsx                      0    0     0 full             ✓
감사조서서식_1100A_계약전 위험평가표 수임 2025.xlsx            1611    0     0 full             ✓
감사조서서식_1100~1300 감사계약.xlsx                        6    0     0 full             ✓
감사조서서식_1101A_계약전 위험평가표 계속 2025.xlsx            1413    0     0 full             ✓
감사조서서식_1300A_독립성준수검토조서 2025.xlsx                  0    0     0 full             ✓
감사조서서식_1300B_개별 감사 참여자의 독립성 준수 확인서 2025.xl        0    0     0 full             ✓
감사조서서식_1300C_산업전문가검토조서.xlsx                       0    0     0 full             ✓
감사조서서식_2100-2700 위험평가 2025.xlsx                 946    0     0 full             ✓
감사조서서식_2110-2 계